# 样本集划分

In [1]:
import os
import torch
import numpy as np
from sklearn.model_selection import train_test_split

# 加载图数据和标签
work_dir = r'D:\博士文件\博士毕业课题材料\维吾尔医药配伍机制量化分析\data'
merged_file = os.path.join(work_dir, 'all_graphs_to_be_predicted.pt')
merged_graphs = torch.load(merged_file, weights_only=False)

# 假设每个图数据都有 y 属性表示标签，并转换为 NumPy 数组
labels = np.array([graph.y.numpy() if isinstance(graph.y, torch.Tensor) else graph.y for graph in merged_graphs])

def random_train_val_split(graphs, labels, val_size=0.3, random_state=10):
    """使用随机划分 train_val_split
    """
    train_graphs, val_graphs, train_labels, val_labels = train_test_split(
        graphs, labels, test_size=val_size, random_state=random_state, shuffle=True)

    return train_graphs, val_graphs, train_labels, val_labels

# 数据集划分
train_graphs, val_graphs, train_labels, val_labels = random_train_val_split(
    merged_graphs, labels, val_size=0.33, random_state=42)

# 计算每个子集中标签中 1 的比例
def calculate_label_proportions(labels):
    proportions = np.mean(labels == 1, axis=0)  # 计算每个标签中 1 的比例
    return proportions

# 计算训练集和验证集中每个标签为 1 的比例
train_proportions = calculate_label_proportions(train_labels)
val_proportions = calculate_label_proportions(val_labels)

# 将标签转换为 torch.Tensor
train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)

# 打印每个子集的大小
print(f"Training set: {len(train_graphs)} graphs")
print(f"Validation set: {len(val_graphs)} graphs")

# 打印每个标签中 1 的比例
print("Proportion of '1's for each label in training set:", train_proportions)
print("Proportion of '1's for each label in validation set:", val_proportions)


Training set: 321 graphs
Validation set: 159 graphs
Proportion of '1's for each label in training set: [0.40809969 0.71962617 0.52336449 0.47663551]
Proportion of '1's for each label in validation set: [0.42138365 0.67295597 0.53459119 0.43396226]


# 多层注意力模型

In [2]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATConv, global_mean_pool
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 设置随机种子
def set_seed(seed):
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # 如果使用GPU

set_seed(42)  # 设置你的随机种子

# GAT 模型定义
class GATModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_heads, dropout_rate=0.3, dosage_weight=1.0):
        super(GATModel, self).__init__()
        self.dosage_weight = dosage_weight  # 用于放大第91号特征的权重
        
        # 增加四层 GAT 注意力机制
        self.layer1 = GATConv(in_dim, hidden_dim, heads=num_heads, dropout=dropout_rate)
        self.layer2 = GATConv(hidden_dim * num_heads, hidden_dim, heads=num_heads, dropout=dropout_rate)
        self.layer3 = GATConv(hidden_dim * num_heads, hidden_dim, heads=num_heads, dropout=dropout_rate)
        self.layer4 = GATConv(hidden_dim * num_heads, hidden_dim, heads=1, dropout=dropout_rate)  # 第四层 GAT
        
        # 全连接层，用于最终输出
        self.fc = nn.Linear(hidden_dim, out_dim)

        # 权重初始化
        self._initialize_weights()

    def _initialize_weights(self):
        for layer in [self.layer1, self.layer2, self.layer3, self.layer4]:
            layer.reset_parameters()  # PyG 官方方法
        self.fc.reset_parameters()


    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # 取出第91号特征并乘以 self.dosage_weight，并限制其放大范围
        x[:, 90] = torch.clamp(x[:, 90] * self.dosage_weight, min=0, max=10)

        # 执行 GAT 层计算并获取注意力权重
        h, attn_weights_1 = self.layer1(x, edge_index, return_attention_weights=True)
        h = torch.relu(h)

        h, attn_weights_2 = self.layer2(h, edge_index, return_attention_weights=True)
        h = torch.relu(h)

        h, attn_weights_3 = self.layer3(h, edge_index, return_attention_weights=True)
        h = torch.relu(h)

        h, attn_weights_4 = self.layer4(h, edge_index, return_attention_weights=True)  # 计算第四层的输出和注意力权重
        
        # 使用全局均值池化汇聚节点信息
        hg = global_mean_pool(h, batch)
        
        # 输出层
        out = self.fc(hg)
        
        # 返回最终输出以及所有的注意力权重
        return out, hg, (attn_weights_1, attn_weights_2, attn_weights_3, attn_weights_4)

# 模型参数设置
in_dim = 91       # 节点特征维度
hidden_dim = 64   # 隐藏层维度
out_dim = 4       # 输出维度，对应5个标签
num_heads = 2     # GAT多头注意力数量
dropout_rate = 0.4
dosage_weight = 1  # 放大剂量参数的权重

# 构建模型
model = GATModel(in_dim, hidden_dim, out_dim, num_heads, dropout_rate, dosage_weight=dosage_weight)
print(model)

GATModel(
  (layer1): GATConv(91, 64, heads=2)
  (layer2): GATConv(128, 64, heads=2)
  (layer3): GATConv(128, 64, heads=2)
  (layer4): GATConv(128, 64, heads=1)
  (fc): Linear(in_features=64, out_features=4, bias=True)
)


# 模型训练

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score
from torch_geometric.loader import DataLoader
from tqdm import tqdm

# 计算每个类别的 pos_weight
num_classes = train_labels.size(1)
pos_counts = train_labels.sum(dim=0)  # 每个类别的正样本数量
neg_counts = train_labels.size(0) - pos_counts  # 每个类别的负样本数量
pos_weight = neg_counts / (pos_counts + 1e-6)

# 损失函数、优化器和调度器
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
learning_rate = 0.0003
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

# 早停机制
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = np.inf
        self.counter = 0

    def check_early_stop(self, val_loss):
        if self.best_loss - val_loss > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

# 打印学习率
def print_learning_rate(optimizer):
    for param_group in optimizer.param_groups:
        print(f"Current Learning Rate: {param_group['lr']}")

# 自定义collate_fn
def custom_collate(batch):
    return batch  # 返回原始batch，不做任何处理

# 数据加载
def create_batches(graphs, labels, batch_size):
    for i, graph in enumerate(graphs):
        graph.y = labels[i]  # 将标签附加到图数据中
    data_loader = DataLoader(graphs, batch_size=batch_size, shuffle=True, collate_fn=custom_collate)
    return data_loader

# 模型评估函数
def evaluate_model(val_loader, model, loss_fn):
    model.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    all_attn_weights = []  # 用于收集所有样本的注意力权重
    with torch.no_grad():
        for batch in val_loader:
            output,_,attn_weights = model(batch)

            # 调整标签形状
            batch_labels = batch.y.view(output.shape)
            loss = loss_fn(output, batch_labels)
            val_loss += loss.item()

            preds = torch.round(torch.sigmoid(output))
            all_preds.append(preds.cpu().numpy())
            all_labels.append(batch_labels.cpu().numpy())
            all_attn_weights.append(attn_weights)  # 收集注意力权重

    avg_val_loss = val_loss / len(val_loader)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    # 计算召回率和F1分数
    recall = recall_score(all_labels, all_preds, average='micro')
    f1 = f1_score(all_labels, all_preds, average='micro')

    return avg_val_loss, recall, f1, all_attn_weights  # 返回注意力权重

# 模型训练过程
def train_model(train_graphs, train_labels, val_graphs, val_labels, model, loss_fn, optimizer, scheduler, num_epochs=50, batch_size=16, early_stopping_patience=3, grad_clip_value=1.0):
    train_loader = create_batches(train_graphs, train_labels, batch_size)
    val_loader = create_batches(val_graphs, val_labels, batch_size)

    early_stopping = EarlyStopping(patience=early_stopping_patience)
    val_losses = []
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            # 前向传播
            output,_, _ = model(batch)  # 注意力权重不需要在训练时保存

            # 调整标签形状
            batch_labels = batch.y.view(output.shape)
            loss = loss_fn(output, batch_labels)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()

            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_value)

            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {avg_loss:.4f}")

        # 验证模型
        val_loss, val_recall, val_f1, attn_weights = evaluate_model(val_loader, model, loss_fn)
        print(f"Epoch {epoch + 1}/{num_epochs}, Validation Loss: {val_loss:.4f}, Validation Recall: {val_recall:.4f}, Validation F1: {val_f1:.4f}")

        # 调整学习率并打印
        scheduler.step(val_loss)
        print_learning_rate(optimizer)

        # 早停检查
        if early_stopping.check_early_stop(val_loss):
            print(f"Early stopping at epoch {epoch + 1}")
            break
           

# 训练模型
train_model(train_graphs, train_labels, val_graphs, val_labels, model, loss_fn, optimizer, scheduler, num_epochs=30, batch_size=16)

Epoch 1/30: 100%|██████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 88.99it/s]


Epoch 1/30, Training Loss: 0.6476
Epoch 1/30, Validation Loss: 0.6487, Validation Recall: 0.5701, Validation F1: 0.5412
Current Learning Rate: 0.0003


Epoch 2/30: 100%|█████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 106.41it/s]


Epoch 2/30, Training Loss: 0.6295
Epoch 2/30, Validation Loss: 0.6446, Validation Recall: 0.5396, Validation F1: 0.5566
Current Learning Rate: 0.0003


Epoch 3/30: 100%|█████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 112.69it/s]


...

Epoch 6/30, Training Loss: 0.5677
Epoch 6/30, Validation Loss: 0.6297, Validation Recall: 0.6829, Validation F1: 0.6162
Current Learning Rate: 0.0003


Epoch 7/30: 100%|█████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 121.14it/s]


Epoch 7/30, Training Loss: 0.5641
Epoch 7/30, Validation Loss: 0.6299, Validation Recall: 0.6098, Validation F1: 0.6126
Current Learning Rate: 0.0003


Epoch 8/30: 100%|██████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 94.79it/s]

Epoch 8/30, Training Loss: 0.5304
Epoch 8/30, Validation Loss: 0.6304, Validation Recall: 0.6250, Validation F1: 0.6003
Current Learning Rate: 0.00015
Early stopping at epoch 8


# 模型保存于加载

In [10]:
import os

model_save_path = r'D:\博士文件\博士毕业课题材料\维吾尔医药配伍机制量化分析\data\gat_model.pth'
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)  # 自动创建目录（如果已存在不报错）

torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")


Model weights saved to D:\博士文件\博士毕业课题材料\维吾尔医药配伍机制量化分析\data\gat_model.pth


# 模型评价

In [7]:
import torch
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support, accuracy_score, confusion_matrix
import pandas as pd
import os
from matplotlib import rcParams

# 设置全局字体为 Arial
rcParams['font.family'] = 'Arial'

def evaluate_model(graphs, labels, model, output_dir, data_name, cpm_id=None):
    model.eval()
    all_outputs = []
    all_labels = []
    all_attn_weights = []

    with torch.no_grad():
        for i, graph in enumerate(graphs):
            output, hg, attn_weights = model(graph)
            all_outputs.append(output.cpu().numpy())
            all_labels.append(labels[i].cpu().numpy())
            all_attn_weights.append(attn_weights)

    final_outputs = np.vstack(all_outputs)
    final_labels = np.vstack(all_labels)

    compute_and_save_metrics(final_labels, final_outputs, output_dir, data_name)

    if cpm_id is not None:
        output_attention_weights(all_attn_weights, graphs, cpm_id, output_dir)

def output_attention_weights(all_attn_weights, graphs, cpm_id, output_dir):
    for i, graph in enumerate(graphs):
        if hasattr(graph, 'cpm_id') and graph.cpm_id == cpm_id:
            attn_weights = all_attn_weights[i]
            attn_weights_1, attn_weights_2, _ = attn_weights

            # 将 attn_weights_1 和 attn_weights_2 转换为 numpy 数组
            attn_weights_1_array = [aw.cpu().numpy() for aw in attn_weights_1]
            attn_weights_2_array = [aw.cpu().numpy() for aw in attn_weights_2]

            # 获取节点名称
            node_names = graph.node_names  # 确保这个属性存在并正确获取

            # 转置第一个数组
            transposed_0 = attn_weights_1_array[0].T  # 现在形状变为 (132, 2)

            # 替换 transposed_0 中的索引为节点名称
            corresponding_node_names = []
            for index_pair in transposed_0:
                name_pair = [node_names[int(idx)] for idx in index_pair]
                corresponding_node_names.append(name_pair)

            # 转换为 NumPy 数组
            corresponding_node_names = np.array(corresponding_node_names)

            # 合并对应的节点名称和权重
            merged_array = np.column_stack((corresponding_node_names, attn_weights_1_array[1]))

            # 保存为 CSV 文件
            np.savetxt(os.path.join(output_dir, f'{cpm_id}_attn_weights_1-多层注意力.csv'), merged_array, delimiter=',', fmt='%s')
            print(f"注意力权重 attn_weights_1 已保存为 {cpm_id}_attn_weights_1.csv")

            # 对 attn_weights_2 进行类似处理（如果需要的话）
            if attn_weights_2_array:
                transposed_2 = attn_weights_2_array[0].T  # 转置第二个数组
                corresponding_node_names_2 = []
                for index_pair in transposed_2:
                    name_pair = [node_names[int(idx)] for idx in index_pair]
                    corresponding_node_names_2.append(name_pair)

                # 转换为 NumPy 数组
                corresponding_node_names_2 = np.array(corresponding_node_names_2)

                # 合并对应的节点名称和权重
                merged_array_2 = np.column_stack((corresponding_node_names_2, attn_weights_2_array[1]))

                # 保存为 CSV 文件
                np.savetxt(os.path.join(output_dir, f'{cpm_id}_attn_weights_2-多层注意力.csv'), merged_array_2, delimiter=',', fmt='%s')
                print(f"注意力权重 attn_weights_2 已保存为 {cpm_id}_attn_weights_2.csv")



# 计算和保存指标的函数（不变）
def compute_and_save_metrics(labels, outputs, output_dir, data_name):
    num_classes = labels.shape[1]
    metrics = {
        'Class': [],
        'Precision': [],
        'Recall': [],
        'F1 Score': [],
        'AUC': [],
        'Accuracy': [],
        'Specificity': []
    }
    roc_data_long_format = {'Class': [], 'Reference': [], 'Predicted': []}
    
    for i in range(num_classes):
        # 使用 Sigmoid 函数将 logits 转换为概率
        probabilities = torch.sigmoid(torch.tensor(outputs))  # 转换为 0~1 概率
        
        # ROC curve
        fpr, tpr, thresholds = roc_curve(labels[:, i], probabilities[:, i].numpy())
        roc_auc = auc(fpr, tpr)
        
        # 保存参考值和预测值到长格式
        for ref, pred in zip(labels[:, i], probabilities[:, i].numpy()):
            roc_data_long_format['Class'].append(f'Class_{i+1}')
            roc_data_long_format['Reference'].append(ref)
            roc_data_long_format['Predicted'].append(pred)
        
        # Precision, Recall, F1, Accuracy, Specificity
        pred_binary = (probabilities[:, i] > 0.5).numpy().astype(int)  # 使用 0.5 作为阈值
        precision, recall, f1, _ = precision_recall_fscore_support(labels[:, i], pred_binary, average='binary')
        accuracy = accuracy_score(labels[:, i], pred_binary)
        
        tn, fp, fn, tp = confusion_matrix(labels[:, i], pred_binary).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # 防止除零
        
        # Save metrics
        metrics['Class'].append(f'Class_{i+1}')
        metrics['Precision'].append(precision)
        metrics['Recall'].append(recall)
        metrics['F1 Score'].append(f1)
        metrics['AUC'].append(roc_auc)
        metrics['Accuracy'].append(accuracy)
        metrics['Specificity'].append(specificity)
    
    # 计算平均值
    avg_metrics = {
        'Class': ['Average'],
        'Precision': [np.mean(metrics['Precision'])],
        'Recall': [np.mean(metrics['Recall'])],
        'F1 Score': [np.mean(metrics['F1 Score'])],
        'AUC': [np.mean(metrics['AUC'])],
        'Accuracy': [np.mean(metrics['Accuracy'])],
        'Specificity': [np.mean(metrics['Specificity'])]
    }
    
    # 添加平均值到 metrics
    for key in metrics:
        metrics[key].append(avg_metrics[key][0])
    
    # 保存长格式的 ROC 数据到 CSV
    roc_df_long = pd.DataFrame(roc_data_long_format)
    roc_df_long.to_csv(os.path.join(output_dir, f'{data_name}_roc_data_多层注意力.csv'), index=False)

    # 保存指标数据到 CSV
    metrics_df = pd.DataFrame(metrics)
    print(metrics_df)
    metrics_df.to_csv(os.path.join(output_dir, f'{data_name}_metrics_多层注意力.csv'), index=False)
    print(f"Metrics and ROC data saved to {output_dir}.")

# 示例调用
work_dir = r'D:\博士文件\博士毕业课题材料\维吾尔医药配伍机制量化分析\data'



# 评估训练集
evaluate_model(train_graphs, train_labels, model, work_dir, "train")
# 评估验证集
evaluate_model(val_graphs, val_labels, model, work_dir, "validation")

# 评估测试集
#evaluate_model(test_graphs, test_labels, model, work_dir, "test-0", cpm_id='CPM05651')

# 评估测试集
#evaluate_model(test_graphs, test_labels, model, work_dir, "test")


     Class  Precision    Recall  F1 Score       AUC  Accuracy  Specificity
0  Class_1   0.666667  0.778626  0.718310  0.838730  0.750779     0.731579
1  Class_2   0.878981  0.597403  0.711340  0.758802  0.651090     0.788889
2  Class_3   0.631579  0.785714  0.700265  0.735041  0.647975     0.496732
3  Class_4   0.654028  0.901961  0.758242  0.827498  0.725857     0.565476
4  Average   0.707814  0.765926  0.722039  0.790018  0.693925     0.645669
Metrics and ROC data saved to D:\博士文件\博士毕业课题材料\维吾尔医药配伍机制量化分析\data.
     Class  Precision    Recall  F1 Score       AUC  Accuracy  Specificity
0  Class_1   0.477612  0.477612  0.477612  0.616483  0.559748     0.619565
1  Class_2   0.707865  0.588785  0.642857  0.591661  0.559748     0.500000
2  Class_3   0.563830  0.623529  0.592179  0.641017  0.540881     0.445946
3  Class_4   0.542857  0.826087  0.655172  0.717552  0.622642     0.466667
4  Average   0.573041  0.629003  0.591955  0.641678  0.570755     0.508044
Metrics and ROC data saved to D:\